# 리포트 8 — 처리 사슬 — 직접파를 죽이고 표적을 세운다

> ECA 로 직접파를 지우고 CFAR 로 문턱을 세운다. **문턱을 어디에 두느냐가 결과를 정하므로** 그 교정을 먼저 적는다.

이 권은 아래 절로 이루어진다. 각 절은 **한 일 · 결과 · 방법 · 재현** 을 자기 앞에 달고 있어, 필요한 절만 따로 읽어도 된다.

| 절 | 무엇을 말하나 | 만든 곳 |
|---|---|---|
| 1 | 수신 → ECA → 거리도플러 → CFAR, 사슬의 형상은 파형이 정한다 | `_parts/51_chain.ipynb` |
| 2 | 탭을 늘리면 환경이 정한 바닥에서 멈추고, 그 대가가 0-도플러 노치다 | `_parts/52_eca.ipynb` |
| 3 ⭐ | 운용 형상에서 경험 Pfa 를 재니 명목값의 1.52~2.66 배였다 | `_parts/53_cfar-calib.ipynb` |
| 4 | 그 배율의 원인은 셀 상관이고, 교정표는 형상마다 다시 재야 한다 | `_parts/54_cfar-why.ipynb` |

⭐ 표시한 절 하나만 읽어도 이 권의 결론은 선다.

숫자는 전부 계산 결과 JSON(원장)에서 주입된다 — 절 끝 «출처» 표가 그 파일과 키다. 원장이 다시 계산되면 빌더를 돌리는 것만으로 본문 숫자가 따라 바뀐다.

이 권에는 별편이 한 편 딸려 있다 — 이 권의 물음을 더 파고들거나 받쳐 주는 글이다.

- [리포트 8-2 «기준채널이 현실이면 얼마를 잃는가»](08_2_two_channel.ipynb) — 8 권의 사슬을 한 줄도 안 고치고, 기준신호를 «송신 파형 그대로» 에서 «잡음·다중경로가 섞인 측정 신호» 로 바꿔 손실을 잰다.

전체 목차는 [reports/README.md](README.md) 이고, 열두 권의 지도는 [리포트 1 «이 연구가 묻는 것과 답한 방식»](01_map.ipynb) 다.


---

## 절 1. 수신 → ECA → 거리도플러 → CFAR, 사슬의 형상은 파형이 정한다



> ### 한 일
> **세 조명원을 하나의 동일한 검출 사슬에 물리고, 사슬의 각 단계가 파형마다 어떤 형상(거리 빈 수 · ECA 탭 수 · 도플러 빈)을 갖는지를 표로 고정했다.**

### 결과
1. 사슬은 네 단계다 — 2채널 수신 → ECA 로 직접파 제거 → 거리-도플러 상관 → CA-CFAR 판정. 각 단계는 앞 단계의 잔류물을 물려받는다.
2. 직접파-에코 비 DNR 은 WiFi 43.0 dB [^1] · LTE 60.0 dB [^2] · 5G 48.9 dB [^3] 다 — 수신단에서 가장 큰 신호이고 2단계가 지울 대상이다.
3. ECA 탭 수는 파형이 정한다 — WiFi 24 [^4] · LTE 14 [^5] · 5G 32 [^6].
4. 도플러 빈은 세 파형 모두 48 [^7]개다(CPI 당 프레임 수). 거리 빈 수와 PRF 는 파형마다 다르고, 아래 표가 그 형상을 한 자리에 모은다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 사슬 구현 | 네 단계가 한 파일에 있다 — `src/passive_process.py:42`(수신) · `:93,124`(ECA) · `:133`(거리-도플러) · `:153`(CA-CFAR) |
| 형상 표 | 선언값을 옮겨 적지 않고 검증 산출물 `outputs/verify_eca.json:meta.setups` 에서 직접 뽑는다 |
| 세 파형 동일 사슬 | 코드 경로가 하나다 — 파형이 바꾸는 것은 형상 파라미터뿐이다 |

### 재현

```bash
cd /workspace/sionna
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/verify_eca.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/viz_report04_detector.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part09_detector.py
```

| | |
|---|---|
| 출력 | `outputs/verify_eca.json` |
| 소요 | ECA 검증 · 그림은 각각 수 분 (GPU 1장) |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [리포트 7 절 1 «상시이면서 내용을 미리 아는 신호는 표준마다…»](07_illuminators.ipynb) | 세 표준의 상시 기준신호와 대역 — 이 편의 형상 표는 그 중 G3(풀로드) 격자에서 뽑은 셋업이다 |
| [리포트 7 절 4 «바이스태틱 거리 분해능은 c/B»](07_illuminators.ipynb) | 거리 규약 $\Delta R_b = c/B_{ref}$ |

---


## 수신 신호가 판정이 되기까지

패시브 검출은 네 단계다. 각 단계는 앞 단계의 잔류물을 물려받는다.

| 단계 | 하는 일 | 코드 |
|---|---|---|
| 1. 수신 | 서베일런스(표적 쪽을 보는 채널) + 레퍼런스(조명원을 직접 받는 채널) 2채널 | `src/passive_process.py:42` |
| 2. ECA | 직접파를 서베일런스에서 투영 제거 | `src/passive_process.py:93,124` |
| 3. 거리-도플러(CAF) | 레퍼런스와 지연 · 도플러 상관 | `src/passive_process.py:133` |
| 4. CA-CFAR | 이웃 셀로 문턱을 세우고 판정 | `src/passive_process.py:153` |


![f1_chain](../outputs/figures/report04_f1_chain.png)

**그림 1.** 수신 신호는 어떤 단계를 거쳐 검출 판정이 되는가?


## 사슬의 형상 — 파형이 정하는 것

직접파는 수신단에서 가장 큰 신호이고, 2단계가 지울 대상이다. 그것이 표적 에코보다 몇 dB 큰지가 DNR 이다. 거리 빈 수와 ECA 탭 수는 파형이 정하고, 도플러 빈은 세 파형 모두 48 [^7]개다.

| 파형 | DNR | ECA 탭 | 거리 빈 | PRF | Δf_d |
|---|---|---|---|---|---|
| WiFi 80MHz | 43.0 dB | 24 | 16 | 1000 Hz | 20.83 Hz |
| LTE 20MHz | 60.0 dB | 14 | 6 | 1000 Hz | 20.83 Hz |
| 5G NR 100MHz | 48.9 dB | 32 | 24 | 2000 Hz | 41.67 Hz |

출처 [^8]

이 표는 G3(풀로드) 점유 격자의 셋업이다(`benchmark/verify_eca.py:505`). 숫자 열 다섯은 점유 체제에 불변이다 — DNR 은 직접파/에코 비라 기하와 반송파별 표적 σ 가 정하고(`benchmark/link_budget.py:86`), 거리 빈·ECA 탭은 표본율이(`benchmark/geometry.py:146`), PRF·Δf_d 는 표본율·프레임 길이·CPI 프레임 수가 정한다(`benchmark/verify_eca.py:107` · `benchmark/run_min_cell.py:74`). 점유가 바꾸는 것은 기준신호의 이름이다.

이 표의 PRF 는 **검출기 프레임률**이다 — 사슬이 한 프레임으로 끊어 읽는 속도이지 조명원의 물리 반복률이 아니다. 5G 상시 SSB 에서 두 양이 얼마나 벌어지고 그것이 도플러를 어디서 접는지는 [리포트 7 절 7 «5G SSB 는 걷는 드론에서 접힌다»](07_illuminators.ipynb) 가 잰다.


## 이 사슬 위에서 무엇이 결정되나

2단계의 소거 깊이와 그 대가는 **절 2** «탭을 늘리면 환경이 정한 바닥에서 멈추고» 가, 4단계 문턱의 눈금은 **절 3** «운용 형상에서 경험 Pfa 를 재니 명목값의…» 가 든다. 3단계가 만드는 응답의 모양은 [리포트 7 절 6 «검출기가 실제로 쓰는 커널 그대로 모호함수를…»](07_illuminators.ipynb) 가 이미 쟀다.

세 파형이 같은 코드 경로를 지나므로, 뒤 편들이 재는 격차는 사슬 차이가 아니라 파형 차이다.

이 사슬의 레퍼런스 채널에는 **송신 파형 그 자체**가 들어간다 — 잡음 0 · 다중경로 0 이라는 상한 가정이다. 그 가정 하나만 풀어 손실을 재는 편이 [리포트 8-2 «기준채널이 현실이면 얼마를 잃는가»](08_2_two_channel.ipynb) 다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 회전 블레이드 산란을 이 사슬에 넣는 조건을 세운다 | 마이크로도플러를 검출 판정에 쓰는 조건이 결정된다 | [리포트 6-6 절 4 «상시 기준신호가 주는 것은 날개끝 확산이 아니…»](06_6_microdoppler-limits.ipynb) |
| 2채널 수신을 실제 X410 캡처로 바꿔 같은 사슬을 돌린다 | 시뮬 사슬과 실측 사슬이 같은 형상 표 위에 선다 | [리포트 10-2 절 4 «X410 의 12-bit ADC 동적범위가 직…»](10_2_robustness.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 8개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/verify_eca.json` | `meta.setups[1].dnr_db` | 42.99 |
| [^2] | `outputs/verify_eca.json` | `meta.setups[2].dnr_db` | 59.99 |
| [^3] | `outputs/verify_eca.json` | `meta.setups[0].dnr_db` | 48.85 |
| [^4] | `outputs/verify_eca.json` | `meta.setups[1].n_taps` | 24 |
| [^5] | `outputs/verify_eca.json` | `meta.setups[2].n_taps` | 14 |
| [^6] | `outputs/verify_eca.json` | `meta.setups[0].n_taps` | 32 |
| [^7] | `outputs/verify_cfar.json` | `meta.M_cpi` | 48 |
| [^8] | `outputs/verify_eca.json` | `meta.setups` | (3행 표) |


---

## 절 2. 탭을 늘리면 환경이 정한 바닥에서 멈추고, 그 대가가 0-도플러 노치다



> ### 한 일
> **ECA 탭 수를 1~96 으로 스윕하며 소거 깊이를 재고, 같은 소거기가 표적의 느린 도플러를 얼마나 함께 지우는지를 속도 문턱으로 환산했다.**

### 결과
1. G3(풀로드) 격자에서 직접파를 기준신호로 합성해 같은 소거기를 걸면 float64 한계까지 내려간다 — 5G 232.3 dB [^9]. 레이 트레이싱으로 푼 챔버 다중경로를 넣으면 56.1 dB [^10] 에서 멈춘다.
2. 그 합성에서 바닥을 정하는 것은 탭 수가 아니라 환경이다 — 탭 1~96 스윕에서 깊이가 포화한다. 직접파를 송신 파형 전체로 합성하면 같은 격자의 깊이가 5G 1.60 dB [^11] 다.
3. 대가는 0-도플러 노치다. 3 dB 손실 지점은 $f_d/\Delta f_d$ = 0.596 [^12] 로 세 파형이 같고, 그 무차원 상수는 M 이 정한다 — M = 16 · 48 · 96 에서 0.613 [^13] · 0.596 [^12] · 0.592 [^14] 다.
4. 프레임 48 [^15]개에서 속도 문턱은 WiFi 0.39 m/s [^16] · LTE 1.10 m/s [^17] · 5G 1.16 m/s [^18] 다 — 그보다 빠른 표적이 무는 노치 손실은 3 dB 아래다. ⚠ 같은 프레임 수에서 $T_{CPI}$ 는 48 ms [^19] · 48 ms [^20] · 24 ms [^21] 로 갈린다.
5. 정적 산란체는 ECA 뒤에서 죽은 파라미터다 — 클러터를 100 [^22]배까지 키워도 SCR 변화폭은 3.5e-09 dB [^23] 다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 소거기 | CPI 1회 최소제곱 사영의 standard ECA — 레퍼런스의 지연 사본이 치는 부분공간에 서베일런스를 통째로 사영한다(`src/passive_process.py:13`) |
| 점유 격자 | G3(풀로드) 한 판이다(`benchmark/verify_eca.py:505`) — 그 격자의 기준신호는 WiFi VHT-LTF [^24] · LTE PRS [^25] · 5G NR-PRS [^26] 라, LTE·5G 는 상시 CRS·SSB 자리에 측위 세션 신호가 선다 |
| 깊이 스윕 | 탭 수 1~96 × (직접파만 / 다중경로 포함) 두 조건. 다중경로는 챔버 장면을 레이 트레이싱으로 푼 것이고 실측이 아니다(출처 RT [^27]). 두 조건의 차이가 «바닥을 무엇이 정하는가» 를 가른다 |
| 노치 환산 | 3 dB 손실 지점을 $f_d/\Delta f_d$ 무차원으로 재고, 파형별 $\lambda$ 로 속도 문턱으로 옮긴다 |
| 클러터 대조 | 정적 산란체 세기를 배수로 키우며 SCR 변화폭을 잰다 |

### 재현

```bash
cd /workspace/sionna
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/verify_eca.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/viz_report04_detector.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part09_detector.py
```

| | |
|---|---|
| 출력 | `outputs/verify_eca.json` |
| 소요 | ECA 검증 · 그림은 각각 수 분 (GPU 1장) |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| **절 1** «수신 → ECA → 거리도플러 → CFAR» | 사슬의 네 단계와 파형별 형상 |

---


## 직접파를 얼마나 지울 수 있는가

탭을 늘리면 소거가 깊어지다가 멈춘다. 멈추는 자리를 정하는 것이 탭 수인지 환경인지를 가르려고 G3(풀로드) 격자에서 두 조건을 나란히 돌렸다.

| 파형 | 직접파(=기준신호로 합성) | 직접파(=기준신호) + 다중경로(포화) |
|---|---|---|
| WiFi | 202.7 dB [^28] | 33.0 dB [^29] |
| LTE | 219.9 dB [^30] | 41.1 dB [^31] |
| 5G | 232.3 dB [^9] | 56.1 dB [^10] |

왼쪽은 float64 산술의 한계이고, 오른쪽이 이 장면의 다중경로가 정하는 바닥이다(실측 채널이 아니라 레이 트레이싱으로 푼 챔버 장면이다). 두 열의 간격이 곧 **환경이 정하는 몫**이다.

두 열 모두 직접파를 기준신호로 합성한 판이다(`benchmark/verify_eca.py:146,147`). 헤드라인 사슬은 직접파를 송신 파형 전체(파일럿+데이터)로 합성하고(`benchmark/run_min_cell.py:164`), 그 판의 **시간영역** 깊이는 WiFi 0.35 dB [^32] · LTE 1.33 dB [^33] · 5G 1.60 dB [^11] 다 — 남은 잔류는 열잡음(var=1)보다 WiFi 31.4 dB [^34] · LTE 61.3 dB [^35] · 5G 44.7 dB [^36] 크다.

⚠ 그 잔류가 RD 맵 어디에 서는지는 파형마다 갈린다. 0-도플러 행의 첨두가 잡음 플로어 위로 서는 것은 LTE 한 파형이고(1.5 dB [^37]), WiFi·5G 는 같은 행에서 -122.3 dB [^38] · -137.7 dB [^39] 다. ⛔ 그 두 수는 크기가 아니라 «이 격자에서는 RD 맵에 서지 않았다» 는 표시로 읽고, 왜 LTE 만 남는지는 이 원장 밖의 물음으로 둔다.

⛔ 비-0도플러 첨두(WiFi -164.8 dB [^40] · LTE -41.0 dB [^41] · 5G -180.2 dB [^42])는 0-도플러 첨두를 상수만큼 평행이동한 값이다 — 원장 9줄 전부에서 두 값의 차가 42.50 dB 이고 줄별 폭은 0.09 dB 다. 잔류가 -23.0 ~ +61.3 dB 로 갈리는 9줄에서 그 차가 같으므로, 이 열이 0-도플러 열에 더해 싣는 것은 «0-도플러 행 세 개를 지웠다» 는 규약이다(`benchmark/verify_eca.py:249`) — 사슬의 운용 마스크는 한 행이다(`benchmark/verify_cfar.py:315`).


![f2_eca_depth](../outputs/figures/report04_f2_eca_depth.png)

**그림 2.** ECA 소거 깊이의 바닥을 정하는 것은 무엇인가?


## 대가 — 0-도플러 노치

ECA 는 지연만 다른 성분을 함께 지운다. 느리게 움직이는 표적은 그 성분과 구분되지 않으므로 같이 깎인다. 3 dB 손실 지점은 $f_d/\Delta f_d$ = 0.596 [^12] 이고 세 파형이 같다 — 속도 문턱은 $\lambda$ 와 $\Delta f_d(=1/T_{CPI})$ 둘이 가른다.

| 파형 | $\lambda$ | $T_{CPI}$ (프레임 48 [^15]개) | 3 dB 속도 문턱 (프레임 48 [^15]개) |
|---|---|---|---|
| WiFi | 0.0575 m [^43] | 48 ms [^19] | 0.39 m/s [^16] |
| LTE | 0.1627 m [^44] | 48 ms [^20] | 1.10 m/s [^17] |
| 5G | 0.0857 m [^45] | 24 ms [^21] | 1.16 m/s [^18] |

⚠ 프레임 수를 48 [^15]개로 고정하면 세 파형의 CPI 가 갈린다 — 프레임 길이가 달라 $T_{CPI}$ 가 48 ms [^19] · 48 ms [^20] · 24 ms [^21] 다. 이 표에서 5G 문턱이 LTE 보다 높은 것은 CPI 가 절반이기 때문이고, $\lambda$ 는 5G(0.0857 m [^45])가 LTE(0.1627 m [^44])보다 짧다.

5G 만 프레임 96개로 잡아 CPI 를 48 ms [^19] 로 맞추면 문턱은 WiFi 0.39 m/s [^16] · 5G 0.58 m/s [^46] · LTE 1.10 m/s [^17] 로 $\lambda$ 순서가 된다. ⛔ 이 표의 세 수는 파형이 정한 물리 문턱이 아니라 프레임 수 규약과 함께 정해진 값이다.


![f3_eca_notch](../outputs/figures/report04_f3_eca_notch.png)

**그림 3.** ECA 가 클러터와 함께 지우는 표적의 속도는 얼마인가?


## 정적 클러터는 ECA 뒤에서 죽은 파라미터다

클러터 세기를 100 [^22]배까지 키워도 SCR 변화폭은 3.5e-09 dB [^23] 다. 소거기가 직접파와 함께 정적 성분을 통째로 가져가기 때문이다.

그래서 이 사슬에서 남는 위협은 정적 클러터가 아니라 **표적을 거쳐 오는 성분**이고, 느린 표적은 노치가 먼저 지운다 — 그 축의 결과는 [리포트 10 절 4 «CPI 를 늘리면 세 파형 모두 블라인드율이…»](10_results.ipynb) 가 든다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 실외에서 잰 채널로 소거 바닥을 다시 잰다 | 환경이 정하는 바닥이 실측 채널에서 몇 dB 인지 확정된다 | `benchmark/verify_eca.py` → [리포트 10-2 절 4 «X410 의 12-bit ADC 동적범위가 직…»](10_2_robustness.ipynb) |
| 노치 폭을 CPI 와 함께 스윕한다 | 느린 표적이 노치 밖으로 나오는 CPI 가 수치로 정해진다 | `benchmark/verify_eca.py` → [리포트 10 절 4 «CPI 를 늘리면 세 파형 모두 블라인드율이…»](10_results.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 38개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^9] | `outputs/verify_eca.json` | `S1_depth_vs_taps[0].rows[12].depth_dpi_db` | 232.3 |
| [^10] | `outputs/verify_eca.json` | `S1_depth_vs_taps[0].rows[12].depth_full_db` | 56.07 |
| [^11] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[6].depth_tx_dpi_db` | 1.603 |
| [^12] | `outputs/verify_eca.json` | `S4_target_loss[1].fd_3db_over_dfd` | 0.5957 |
| [^13] | `outputs/verify_eca.json` | `S4_target_loss[0].fd_3db_over_dfd` | 0.6129 |
| [^14] | `outputs/verify_eca.json` | `S4_target_loss[2].fd_3db_over_dfd` | 0.5919 |
| [^15] | `outputs/verify_eca.json` | `S4_target_loss[4].M` | 48 |
| [^16] | `outputs/verify_eca.json` | `S4_target_loss[4].v_3db_ms` | 0.3906 |
| [^17] | `outputs/verify_eca.json` | `S4_target_loss[7].v_3db_ms` | 1.104 |
| [^18] | `outputs/verify_eca.json` | `S4_target_loss[1].v_3db_ms` | 1.163 |
| [^19] | `outputs/verify_eca.json` | `S4_target_loss[4].T_cpi_ms` | 48 |
| [^20] | `outputs/verify_eca.json` | `S4_target_loss[7].T_cpi_ms` | 48 |
| [^21] | `outputs/verify_eca.json` | `S4_target_loss[1].T_cpi_ms` | 24 |
| [^22] | `outputs/verify_eca.json` | `S5_clutter_dead.sweep[3].scale` | 100 |
| [^23] | `outputs/verify_eca.json` | `S5_clutter_dead.scr_span_db` | 3.539e-09 |
| [^24] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[7].ref_name` | VHT-LTF |
| [^25] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[8].ref_name` | PRS |
| [^26] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[6].ref_name` | NR-PRS |
| [^27] | `outputs/verify_eca.json` | `meta.setups[0].clutter_src` | RT |
| [^28] | `outputs/verify_eca.json` | `S1_depth_vs_taps[1].rows[12].depth_dpi_db` | 202.7 |
| [^29] | `outputs/verify_eca.json` | `S1_depth_vs_taps[1].rows[12].depth_full_db` | 33.01 |
| [^30] | `outputs/verify_eca.json` | `S1_depth_vs_taps[2].rows[12].depth_dpi_db` | 219.9 |
| [^31] | `outputs/verify_eca.json` | `S1_depth_vs_taps[2].rows[12].depth_full_db` | 41.12 |
| [^32] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[7].depth_tx_dpi_db` | 0.3466 |
| [^33] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[8].depth_tx_dpi_db` | 1.333 |
| [^34] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[7].resid_over_noise_db` | 31.45 |
| [^35] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[8].resid_over_noise_db` | 61.28 |
| [^36] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[6].resid_over_noise_db` | 44.71 |
| [^37] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[8].rd_zerodop_peak_over_nfloor_db` | 1.499 |
| [^38] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[7].rd_zerodop_peak_over_nfloor_db` | -122.3 |
| [^39] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[6].rd_zerodop_peak_over_nfloor_db` | -137.7 |
| [^40] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[7].rd_offzero_peak_over_nfloor_db` | -164.8 |
| [^41] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[8].rd_offzero_peak_over_nfloor_db` | -41.02 |
| [^42] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[6].rd_offzero_peak_over_nfloor_db` | -180.2 |
| [^43] | `outputs/verify_eca.json` | `S4_target_loss[4].lam_m` | 0.05754 |
| [^44] | `outputs/verify_eca.json` | `S4_target_loss[7].lam_m` | 0.1627 |
| [^45] | `outputs/verify_eca.json` | `S4_target_loss[1].lam_m` | 0.08565 |
| [^46] | `outputs/verify_eca.json` | `S4_target_loss[2].v_3db_ms` | 0.5777 |


---

## 절 3. 운용 형상에서 경험 Pfa 를 재니 명목값의 1.52~2.66 배였다



> ### 한 일
> **운용 형상의 검출 사슬에서 거리-도플러 맵을 대량으로 다시 만들어 경험적 오경보율을 세고, 세 파형의 CFAR 문턱을 그 측정값에 맞춰 교정했다.**

### 결과
1. GPU 2717 s [^47] 동안 파형·모드마다 거리-도플러 맵 10,000 [^48]장을 돌려 경험 Pfa 를 측정했다.
2. 검출기 구현의 눈금을 먼저 확정했다 — 문턱 상수는 이론값과 상대오차 7.6e-16 [^49] 안에서 같고, 이상적 백색 맵 500,000 [^50]장(셀 564,000,000 [^51]개)에서 경험/명목 = 0.997 [^52] 다.
3. 운용 형상(CPI 프레임 48 [^53] · `g2x2_t6x6` · 0-도플러 마스크 1 [^54]빈)에서 명목 1e-04 [^55] 를 주면 WiFi 1.53 [^56]배 · LTE 2.66 [^57]배 · 5G 1.52 [^58]배로 울린다.
4. 그 형상의 교정표를 만들었다 — 경험 1e-04 [^55] 를 얻는 명목값은 WiFi 6.27e-05 [^59] · LTE 2.90e-05 [^60] · 5G 6.46e-05 [^61] 다.
5. 선행 census 16 [^62]편 · 전문 198 [^63]쪽에서 `CFAR` 와 `false alarm` 이 모두 0회인 논문이 13 [^64]편이고, 검출을 주장한 논문은 1 [^65]편이다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 경험 Pfa | 파형·명목값마다 거리-도플러 맵 10,000 [^48]장에 CA-CFAR 를 걸어 오경보 셀을 세었다 (GPU, 2717 s [^47]) |
| 문턱 상수 | CA-CFAR α 를 이론식과 대조했다 — 상대오차 7.6e-16 [^49] · 훈련셀 264 [^66]개 |
| 측정 구간 | 명목 1e-06 [^67] ~ 1e-02 [^68] 아홉 점. 그 밖의 운용점은 외삽이라 표에서 뺀다 |
| 교정표 | 측정한 명목–경험 곡선을 역보간한다. 자유공간 기하는 형상이 달라 `src/freespace_detect.py:711` 이 거기서 다시 잰다 |
| 왜 통제 시뮬레이션인가 | 오경보율을 명목값과 대조하려면 같은 배경을 수만 번 다시 만들어 세어야 한다. 실외 실측은 배경을 주어진 대로 받는다 |

### 재현

```bash
cd /workspace/sionna
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/verify_cfar.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/viz_report04_detector.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part09_detector.py
```

| | |
|---|---|
| 출력 | `outputs/verify_cfar.json`, `outputs/prior_census.json` |
| 소요 | CFAR 측정이 2717 s [^47] (GPU 1장) |
| 비고 | 맵 수는 `--maps` / `--white` 로 줄인다. 줄이면 신뢰구간이 넓어진다. |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| **절 1** «수신 → ECA → 거리도플러 → CFAR» | 사슬의 네 단계와 파형별 형상 |

---


## 왜 배경을 다시 만드는가

⭐ 오경보율을 명목값과 대조하려면 같은 배경을 수만 번 다시 만들어 세어야 한다. 통제 시뮬레이션이 그 일을 한다.

선행 census 16 [^62]편 · 전문 198 [^63]쪽에서 `CFAR` 와 `false alarm` 이 모두 0회인 논문이 13 [^64]편이고, 검출을 주장한 논문은 1 [^65]편이다. OpenISAC(`arXiv:2601.03535v2`, preprint)은 전문 16 [^69]쪽에서 `CFAR` 0 [^70]회 · `false alarm` 0 [^71]회 · `detection probability` 0 [^72]회다.


## 눈금부터 확정한다

이 절의 모든 수는 **운용 형상** 하나에서 나온다 — DPI+ECA · 운용 거리창 · CPI 프레임 48 [^53] · 훈련창 `g2x2_t6x6` · 0-도플러 마스크 1 [^54]빈.

검출기 구현이 먼저 맞아야 배율을 사슬 탓으로 돌릴 수 있다. 문턱 상수는 이론값과 상대오차 7.6e-16 [^49] 안에서 같고, 잡음 추정/실제 전력 = 1.000 [^73] 다. 이상적 백색 맵 500,000 [^50]장에서 경험/명목 = 0.997 [^52] 로 눈금이 1 에 선다.


![f4_pfa](../outputs/figures/report04_f4_pfa.png)

**그림 4.** 명목 Pfa 를 요구하면 실제로는 몇 배가 울리는가?


## 운용 형상 교정표

운용 명목값은 1e-04 [^55] 다. 왼쪽 열이 그 값에서 측정된 배율이고, 오른쪽 열이 교정된 명목값이다.

| 파형 | 명목 1e-4 에서 경험/명목 | 경험 1e-4 를 얻을 명목 Pfa |
|---|---|---|
| WiFi | 1.53 [^56]배 | 6.27e-05 [^59] |
| LTE | 2.66 [^57]배 | 2.90e-05 [^60] |
| 5G | 1.52 [^58]배 | 6.46e-05 [^61] |

세 파형의 배율이 서로 다르다. 교정이 셋을 같은 실제 오경보율 위에 올린다.

`src/passive_process.py:283` 이 이 JSON 을 읽고, `pfa_nominal_for()`(`src/passive_process.py:338`)가 파형별 명목값을 돌려준다.


## 이 표를 읽는 곳

`src/experiment_detection.py:358` 과 `src/experiment_x410.py:175` 가 `src/passive_process.py:283` 을 거쳐 이 표를 읽는다.

교정이 없으면 세 파형 비교가 서로 다른 실제 오경보율 위에서 이뤄진다. 그 위에서 선 비교가 [리포트 9 절 4 «자유공간 형상에서 문턱을 다시 재니 세 밴드가 SNR90 하나를 공유한다»](09_observability.ipynb) 다.

이 표는 **레퍼런스 채널이 이상적일 때**의 형상에서 잰 값이다. 레퍼런스가 오염되면 거리-도플러 맵의 통계가 달라져 이 교정이 그대로 서지 않는다 — 그 형상에서 팔마다 경험 Pfa 를 다시 센 것이 [리포트 8-2 «기준채널이 현실이면 얼마를 잃는가»](08_2_two_channel.ipynb) 다.

배율이 왜 1 이 아닌지, 이 표가 어디까지 쓰이는지는 **절 4** «그 배율의 원인은 셀 상관이고, 교정표는 형상마다 다시 재야 한다» 가 든다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 실외 클러터 배경 위에서 같은 Pfa 스윕을 돌린다 | 교정 배율이 배경에 따라 얼마나 움직이는지 수치로 확정된다 | `benchmark/verify_cfar.py` → [리포트 10-2 절 4 «X410 의 12-bit ADC 동적범위가 직…»](10_2_robustness.ipynb) |
| 맵 수를 한 자릿수 올려 명목 1e-06 [^74] 구간까지 측정한다 | 저 Pfa 운용점의 교정값이 측정 구간 안으로 들어온다 | `benchmark/verify_cfar.py --maps` · `calib_op_mask1.points` |
| 표적 σ 를 앵커 위에서 읽어 Pd 절대값을 다시 푼다 | Pd 절대값이 교정된 Pfa 와 같은 근거 위에 선다 | [리포트 10 절 2 «앵커 σ 위의 R90 은 비교가능 12칸에서…»](10_results.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 28개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^47] | `outputs/verify_cfar.json` | `meta.runtime_s` | 2717 |
| [^48] | `outputs/verify_cfar.json` | `meta.n_maps_chain` | 10000 |
| [^49] | `outputs/verify_cfar.json` | `alpha_audit.g2x2_t6x6.rel_err` | 7.581e-16 |
| [^50] | `outputs/verify_cfar.json` | `meta.n_maps_white` | 500000 |
| [^51] | `outputs/verify_cfar.json` | `white.48x24.rows[89].cells` | 564000000 |
| [^52] | `outputs/verify_cfar.json` | `white.48x24.rows[89].ratio` | 0.9966 |
| [^53] | `outputs/verify_cfar.json` | `meta.M_cpi` | 48 |
| [^54] | `outputs/verify_cfar.json` | `meta.zd_mask_operational` | 1 |
| [^55] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.op.rows[89].pfa_nom` | 0.0001 |
| [^56] | `outputs/verify_cfar.json` | `chain.WiFi80.dpi_eca.op.rows[89].ratio` | 1.531 |
| [^57] | `outputs/verify_cfar.json` | `chain.LTE20.dpi_eca.op.rows[89].ratio` | 2.663 |
| [^58] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.op.rows[89].ratio` | 1.521 |
| [^59] | `outputs/verify_cfar.json` | `chain.WiFi80.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed` | 6.27e-05 |
| [^60] | `outputs/verify_cfar.json` | `chain.LTE20.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed` | 2.905e-05 |
| [^61] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed` | 6.46e-05 |
| [^62] | `outputs/prior_census.json` | `meta.n_papers` | 16 |
| [^63] | `outputs/prior_census.json` | `counts.total_pages` | 198 |
| [^64] | `outputs/prior_census.json` | `counts.zero_cfar_and_falsealarm` | 13 |
| [^65] | `outputs/prior_census.json` | `counts.claims_detection` | 1 |
| [^66] | `outputs/verify_cfar.json` | `alpha_audit.g2x2_t6x6.N_interior` | 264 |
| [^67] | `outputs/verify_cfar.json` | `meta.pfa_nominal[8]` | 1e-06 |
| [^68] | `outputs/verify_cfar.json` | `meta.pfa_nominal[0]` | 0.01 |
| [^69] | `outputs/prior_census.json` | `papers[14].pages` | 16 |
| [^70] | `outputs/prior_census.json` | `papers[14].terms.cfar` | 0 |
| [^71] | `outputs/prior_census.json` | `papers[14].terms.false_alarm` | 0 |
| [^72] | `outputs/prior_census.json` | `papers[14].terms.detection_probability` | 0 |
| [^73] | `outputs/verify_cfar.json` | `alpha_audit.g2x2_t6x6.noise_est_over_power` | 1 |
| [^74] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.calib_op_mask1.points[4].pfa_target_emp` | 1e-06 |


---

## 절 4. 그 배율의 원인은 셀 상관이고, 교정표는 형상마다 다시 재야 한다



> ### 한 일
> **명목과 경험 사이의 배율을 만드는 항을 대조군 두 종으로 하나씩 꺼 확정하고, 그 배율이 형상에 얼마나 의존하는지를 창 폭을 바꿔 재었다.**

### 결과
1. CA-CFAR 는 훈련셀이 서로 독립이라고 가정한다. 사슬은 slow-time Hann 창으로 도플러축 셀을 묶는다 — Hann 을 rect 창으로 바꾸면 5G 배율이 0.96 [^75] 로 내려온다.
2. 거리축 항까지 끄면 1.02 [^76] 로 **잡음 맵**의 눈금이 1 로 돌아온다 — 두 항이 되돌리는 구간은 잡음 맵까지(+0.87 dB)다. 운용 형상 전체 사슬의 1.52 [^77] 는 잡음 맵(1.25 [^78])보다 ×1.22(+0.87 dB) 크고, 대조군 셋은 모두 `mode="noise"` 에서 돌았다(`benchmark/verify_cfar.py:695`).
3. 형상이 배율을 정한다 — 같은 파형이 운용 창에서 1.52 [^77]배, 넓은 창(256 [^79] 빈)에서 47.70 [^80]배다.
4. 그래서 교정표는 형상마다 다시 잰다. `check_detector_config()`(`src/passive_process.py:383`)가 거리창과 0-도플러 마스크 두 조건을 강제한다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 대조군 2종 | slow-time Hann 제거(도플러축) · 백색화 정합필터(거리축). 하나씩 끄고 배율이 어디로 가는지를 본다 |
| 사다리 읽기 | 이상적 백색 맵 → 잡음 맵 → 항을 하나씩 뺀 대조군 → 전체 사슬 순서로 읽으면 각 항의 몫이 갈린다 |
| 형상 의존 | 운용 창과 넓은 창(256 [^79] 빈)에서 같은 파형을 다시 재 배율의 형상 의존을 크기로 적는다 |
| 두 형상의 쓰임 | 운용 형상의 표는 챔버 기하가 읽고, 자유공간 기하는 `src/freespace_detect.py:711` 이 자기 형상에서 다시 잰다 |

### 재현

```bash
cd /workspace/sionna
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/verify_cfar.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/viz_report04_detector.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part09_detector.py
```

| | |
|---|---|
| 출력 | `outputs/verify_cfar.json` |
| 소요 | CFAR 측정이 2717 s [^81] (GPU 1장) |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| **절 3** «운용 형상에서 경험 Pfa 를 재니 명목값의…» | 운용 형상에서 잰 경험 Pfa 와 교정표 |

---


## 원인 — 셀 상관

CA-CFAR 는 훈련셀이 서로 독립이라고 가정한다. 사슬은 slow-time Hann 창으로 도플러축 셀을 묶고, 정합필터로 거리축 셀을 묶는다. 대조군이 그 두 항을 **잡음 맵에 대해** 원인으로 확정한다(5G NR, 명목 1e-4).

⚠ 아래 표에서 가운데 네 줄은 모두 `mode="noise"` 판이고(`benchmark/verify_cfar.py:695`), 마지막 줄만 DPI+ECA 를 지난다 — 그 단계를 끄는 대조군은 이 원장 밖이다.

| 조건 | 경험/명목 |
|---|---|
| 이상적 백색 맵 | 0.997 [^82] |
| 잡음 맵 (Hann + 정합필터) | 1.25 [^78] |
|   └ Hann 제거 (rect 창) | 0.96 [^75] |
|   └ 백색화 정합필터 (거리축 평탄) | 1.25 [^83] |
|   └ 둘 다 제거 | 1.02 [^76] |
| 전체 사슬 (직접파 + ECA) | 1.52 [^77] |


![f5_cause](../outputs/figures/report04_f5_cause.png)

**그림 5.** 명목과 경험 사이의 배율을 만드는 것은 무엇인가?


## 셀 상관은 어느 검출기에나 있다 — 그래서 대조군이 필요하다

«상관이 있다» 만으로는 원인 지목이 서지 않는다. 항을 하나씩 꺼서 눈금이 1 로 돌아오는 것을 보여야 확정된다. Hann 을 rect 로 바꾸면 0.96 [^75], 백색화 정합필터까지 끄면 1.02 [^76] 다.

그 사다리가 위 표다. «둘 다 제거»(1.02 [^76])와 «잡음 맵»(1.25 [^78]) 사이의 +0.87 dB 가 두 항의 몫이고, 백색 맵(0.997 [^82])에서 전체 사슬(1.52 [^77])까지의 전체 초과는 +1.84 dB 다.

⛔ 잡음 맵에서 전체 사슬까지의 +0.87 dB(×1.22, 전체 초과의 47 %)는 대조군 셋이 지나지 않은 DPI+ECA 단계 쪽에 있다 — 셋 다 `mode="noise"` 로 돌았다(`benchmark/verify_cfar.py:695`). 표본은 히트 1,406 [^84] → 1,716 [^85], 셀 11,280,000 [^86] 개다.

원장이 그 단계에 대해 적어 두는 것은 두 수다 — 거리축 lag-1 상관이 잡음 맵 0.07 [^87] 에서 전체 사슬 0.53 [^88] 로 오르고, 2D 유효독립분율이 0.40 [^89] → 0.10 [^90] 로 떨어진다. ⚠ 그 항을 끄는 대조군이 이 원장에 없으므로 원인 지목은 여기서 멈춘다.


## 형상 규약 — 교정표가 성립하는 조건

거리창은 ECA 탭 안에 두고, 0-도플러 행 1 [^91]개를 마스킹한다. `check_detector_config()`(`src/passive_process.py:383`)가 두 조건을 검사한다.

| 파형 | 운용 창 | 넓은 창 |
|---|---|---|
| WiFi | 1.53 [^92]배 | 41.1 [^93]배 |
| LTE | 2.66 [^94]배 | 58.8 [^95]배 |
| 5G | 1.52 [^77]배 | 47.7 [^80]배 |

창을 256 [^79] 빈으로 넓히면 배율이 두 자릿수가 된다. 교정표는 운용 창 형상에서 측정한 값이다.


## 어느 형상의 교정표가 어디에 쓰이나

| 형상 | 무엇을 재나 | 재는 코드 | 그 값을 읽는 곳 |
|---|---|---|---|
| 운용 형상 — CPI 프레임 48 [^96] · `g2x2_t6x6` · 마스크 1 [^91]빈 · 운용 거리창 | 이 부의 교정표 | `benchmark/verify_cfar.py` | `src/experiment_detection.py:358` · `src/experiment_x410.py:175` |
| 자유공간 형상 — 모드별 프레임 수 · 자유공간 거리창 · 0-도플러 가드 | 자유공간 명목 Pfa | `src/freespace_detect.py:711` | `src/experiment_freespace_range.py:206` |

[리포트 9 절 4 «자유공간 형상에서 문턱을 다시 재니 세 밴드가…»](09_observability.ipynb) 가 싣는 명목 Pfa 는 둘째 줄에서 나온 수다 — 이 부의 교정표와 형상이 달라 값도 다르다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 훈련셀에서도 0-도플러 행을 빼는 CFAR 변형을 만든다 | 넓은 창에서 마스크 폭 3 이 만드는 배율 0.65 [^97]배가 마스크 폭과 분리된다 | `src/passive_process.py:352` |
| 표적모형 민감도를 이 배율 위에서 다시 푼다 | 세 표적모형의 절대 소요이득이 경험 Pfa 위에 선다 — CA-CFAR 문턱은 세 팔에 같은 오프셋을 주므로 모형 간 차이는 문턱 규약에 불변이다 | [리포트 10-2 절 2 «평판·큐브·우리 격자를 같은 동작점에서 갈아끼…»](10_2_robustness.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 23개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^75] | `outputs/verify_cfar.json` | `control_rect_window_NR100.op.rows[89].ratio` | 0.9619 |
| [^76] | `outputs/verify_cfar.json` | `control_whitened_mf_rect_NR100.op.rows[89].ratio` | 1.02 |
| [^77] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.op.rows[89].ratio` | 1.521 |
| [^78] | `outputs/verify_cfar.json` | `chain.NR100.noise.op.rows[89].ratio` | 1.246 |
| [^79] | `outputs/verify_cfar.json` | `meta.n_range_wide` | 256 |
| [^80] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.wide.rows[89].ratio` | 47.7 |
| [^81] | `outputs/verify_cfar.json` | `meta.runtime_s` | 2717 |
| [^82] | `outputs/verify_cfar.json` | `white.48x24.rows[89].ratio` | 0.9966 |
| [^83] | `outputs/verify_cfar.json` | `control_whitened_mf_NR100.op.rows[89].ratio` | 1.246 |
| [^84] | `outputs/verify_cfar.json` | `chain.NR100.noise.op.rows[89].hits` | 1406 |
| [^85] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.op.rows[89].hits` | 1716 |
| [^86] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.op.rows[89].cells` | 11280000 |
| [^87] | `outputs/verify_cfar.json` | `chain.NR100.noise.whiteness.rho_range[0]` | 0.07052 |
| [^88] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.whiteness.rho_range[0]` | 0.534 |
| [^89] | `outputs/verify_cfar.json` | `chain.NR100.noise.whiteness.eff_indep_frac_2d` | 0.3991 |
| [^90] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.whiteness.eff_indep_frac_2d` | 0.0987 |
| [^91] | `outputs/verify_cfar.json` | `meta.zd_mask_operational` | 1 |
| [^92] | `outputs/verify_cfar.json` | `chain.WiFi80.dpi_eca.op.rows[89].ratio` | 1.531 |
| [^93] | `outputs/verify_cfar.json` | `chain.WiFi80.dpi_eca.wide.rows[89].ratio` | 41.14 |
| [^94] | `outputs/verify_cfar.json` | `chain.LTE20.dpi_eca.op.rows[89].ratio` | 2.663 |
| [^95] | `outputs/verify_cfar.json` | `chain.LTE20.dpi_eca.wide.rows[89].ratio` | 58.8 |
| [^96] | `outputs/verify_cfar.json` | `meta.M_cpi` | 48 |
| [^97] | `outputs/verify_cfar.json` | `chain.LTE20.dpi_eca.wide.rows[90].ratio` | 0.6538 |
